# Neural codecs with ESPnet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Courses/CMUSpeechTechnology26S/neural_codec.ipynb) [![neural_codec](https://github.com/espnet/notebook/actions/workflows/neural_codec.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/neural_codec.yml)

Encode and decode speech with three pretrained neural codecs, listen to what
each keeps, drop streams to see the bitrate trade, and score the result with
VERSA. The last section repeats it on singing voice.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet codec recipes](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/codec1)
- [VERSA](https://github.com/wavlab-speech/versa)


## Install

We use the inference install of ESPnet rather than the full one: it carries no
training stack, so it is much smaller and much faster to install. VERSA is the
evaluation toolkit used further down.


We need VERSA to support today's evaluation.


In [ ]:
%pip install -q "espnet==202610.post1"
!git clone https://github.com/wavlab-speech/versa
!cd versa && pip install .
# the snr_related config scores with signal_metric, whose metrics are
# VERSA's optional extras: without these three the scorer refuses the
# config rather than dropping the metric
!pip install ci_sdr fast_bss_eval mir_eval
!pip install espnet_model_zoo

!pip install transformers

# IMPORTANT NOTE: due to the recent default change in colab, we need to restart the session to make the numpy(1.23.5) work as expected
# Go to `Runtime` and select `Restart Session`

Data download.


In [ ]:
!git clone https://github.com/ftshijt/versa_demo_egs

## Neural Codec
Neural codecs represent a popular approach nowadays to audio compression and reconstruction, leveraging deep learning to achieve high-quality audio with significantly lower bitrates than traditional methods. In this tutorial, we will explore neural codec technology and demonstrate its application across three key scenarios:

- Different models: Comparing performance across state-of-the-art neural codec architectures (EnCodec, SoundStream, DAC)
- Different data domains: Testing robustness across diverse audio types (speech, music, environmental sounds)

Neural codecs fundamentally differ from traditional audio codecs by replacing hand-crafted signal processing techniques with learned neural representations. Instead of using predefined algorithms, these systems learn optimal compression strategies directly from data but it also presents certain risks of robustness.

This tutorial will provide hands-on examples of neural codec inference, evaluation metrics for comparing performance. By the end, you'll understand how to select and utilize the appropriate neural codec for your specific audio processing needs.


## 1. Neural Codec Models

In today's lecture, we mainly explore three types of codec models, namely
Soundstream, Encodec, and DAC. While Soundstream is one of the earliest attempts in neural codecs with residual vector quantization, Encodec and DAC extends it with a few architectural updates, such as discriminators and training losses. Please refer to their paper for detilas as follows:
- [Soundstream](https://arxiv.org/abs/2107.03312)
- [Encodec](https://arxiv.org/abs/2210.13438)
- [DAC](https://arxiv.org/pdf/2306.06546)

In their original paper, the three codecs are presented with several different
setups, including the training data, training framework, and data preprocessing. Today, we use ESPnet-Codec to evaluate the three models in a controlled setting to see their difference. Specifically, we use only [LibriTTS](https://www.openslr.org/60/) to train the three codec models, with a common share to data preprocessing, training data, and the training framework.


### 1.1 Model Setup


In [ ]:
import torch
import numpy
from espnet2.bin.gan_codec_inference import AudioCoding

soundstream_16k = AudioCoding.from_pretrained("espnet/libritts_soundstream16k")
dac_16k = AudioCoding.from_pretrained("espnet/libritts_dac_16k")
encodec_16k = AudioCoding.from_pretrained("espnet/libritts_encodec_16k")

### 1.2 Encoding and Decoding


In [ ]:
import soundfile as sf
import torch
import numpy as np
import librosa.display
from IPython.display import display, Audio
import matplotlib.pyplot as plt


speech, sr = sf.read("versa_demo_egs/examples/normal_speech/codec/gt/1.wav")
speech = speech.astype(np.float32)
soundstream_codec = soundstream_16k(speech, encode_only=True)["codes"]
dac_codec = dac_16k(speech, encode_only=True)["codes"]
encodec_codec = encodec_16k(speech, encode_only=True)["codes"]

soundstream_speech = soundstream_16k.decode(soundstream_codec)["resyn_audio"].squeeze(0).cpu().numpy()
dac_speech = dac_16k.decode(dac_codec)["resyn_audio"].squeeze(0).cpu().numpy()
encodec_speech = encodec_16k.decode(encodec_codec)["resyn_audio"].squeeze(0).cpu().numpy()

print("--" * 10 + " Original Speech " + "--" * 10)
display(Audio(speech, rate=sr))
librosa.display.waveshow(speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Soundstream " + "--" * 10)
display(Audio(soundstream_speech, rate=sr))
librosa.display.waveshow(soundstream_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from DAC " + "--" * 10)
display(Audio(dac_speech, rate=sr))
librosa.display.waveshow(dac_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Encodec " + "--" * 10)
display(Audio(encodec_speech, rate=sr))
librosa.display.waveshow(encodec_speech, sr=sr, color="blue")
plt.show()

sf.write("soundstream_speech_eg.wav", np.ravel(soundstream_speech), sr)
sf.write("encodec_speech_eg.wav", np.ravel(encodec_speech), sr)
sf.write("dac_speech_eg.wav", np.ravel(dac_speech), sr)

### 1.3 Use Less Streams for Decoding


In [ ]:
import torch
import soundfile as sf
import numpy as np
import librosa.display
from IPython.display import display, Audio
import matplotlib.pyplot as plt

# Set the number of streams/levels (a integer from 1 to 32)
NUM_STREAMS=1

speech, sr = sf.read("versa_demo_egs/examples/normal_speech/codec/gt/1.wav")
speech = speech.astype(np.float32)
soundstream_codec = soundstream_16k(speech, encode_only=True)["codes"][:NUM_STREAMS]
dac_codec = dac_16k(speech, encode_only=True)["codes"][:NUM_STREAMS]
encodec_codec = encodec_16k(speech, encode_only=True)["codes"][:NUM_STREAMS]


soundstream_speech = soundstream_16k.decode(soundstream_codec)["resyn_audio"].squeeze(0).cpu().numpy()
dac_speech = dac_16k.decode(dac_codec)["resyn_audio"].squeeze(0).cpu().numpy()
encodec_speech = encodec_16k.decode(encodec_codec)["resyn_audio"].squeeze(0).cpu().numpy()

print("--" * 10 + " Original Speech " + "--" * 10)
display(Audio(speech, rate=sr))
librosa.display.waveshow(speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Soundstream " + "--" * 10)
display(Audio(soundstream_speech, rate=sr))
librosa.display.waveshow(soundstream_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from DAC " + "--" * 10)
display(Audio(dac_speech, rate=sr))
librosa.display.waveshow(dac_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Encodec " + "--" * 10)
display(Audio(encodec_speech, rate=sr))
librosa.display.waveshow(encodec_speech, sr=sr, color="blue")
plt.show()

# save speech for further usage
sf.write("soundstream_speech_eg-1code.wav", np.ravel(soundstream_speech), sr)
sf.write("encodec_speech_eg-1code.wav", np.ravel(encodec_speech), sr)
sf.write("dac_speech_eg-1code.wav", np.ravel(dac_speech), sr)

In [ ]:
# Use VERSA to evaluate the reconstructed speech

import math
import json

# This time we use SCP option to load the wav
with open("test_wav.scp", "w") as test_wavscp, open("ref_wav.scp", "w") as ref_wavscp:

  print(
      "soundstream_speech_eg soundstream_speech_eg.wav\n"
      "encodec_speech_eg encodec_speech_eg.wav\n"
      "dac_speech_eg dac_speech_eg.wav\n"
      "soundstream_speech_eg_1code soundstream_speech_eg-1code.wav\n"
      "encodec_speech_eg_1code encodec_speech_eg-1code.wav\n"
      "dac_speech_eg_1code dac_speech_eg-1code.wav", file = test_wavscp, end=""
  )

  print(
      "soundstream_speech_eg versa_demo_egs/examples/normal_speech/codec/gt/1.wav\n"
      "encodec_speech_eg versa_demo_egs/examples/normal_speech/codec/gt/1.wav\n"
      "dac_speech_eg versa_demo_egs/examples/normal_speech/codec/gt/1.wav\n"
      "soundstream_speech_eg_1code versa_demo_egs/examples/normal_speech/codec/gt/1.wav\n"
      "encodec_speech_eg_1code versa_demo_egs/examples/normal_speech/codec/gt/1.wav\n"
      "dac_speech_eg_1code versa_demo_egs/examples/normal_speech/codec/gt/1.wav", file = ref_wavscp, end=""
  )


# VERSA ships many metric configurations under versa/egs; swap the
# one below for whichever set you want to compare
!python -m versa.bin.scorer \
    --score_config versa/egs/separate_metrics/snr_related.yaml \
    --gt test_wav.scp \
    --pred ref_wav.scp \
    --output_file test_result


!cat test_result

print("*" * 10, "Better Format", "*" * 10)
# We can make it easier to visualize as follows:
import ast, json
data = []
with open("test_result", 'r') as infile:
    for line in infile:
        line = line.strip()
        if not line:
            continue  # Skip empty lines
        try:
            line = line.replace("inf", "Infinity").replace("'", '"')
            # line = line.replace("'", '"')
            record = json.loads(line)
            # Round float values to two decimals.
            for key, value in record.items():
                if isinstance(value, float):
                    record[key] = round(value, 2)
            # calculate utterance-cer/wer
            if "owsm_hyp_text" in record.keys():
                record["owsm_cer"] = (
                      record["owsm_cer_delete"] +
                      record["owsm_cer_replace"] +
                      record["owsm_cer_insert"]
                  ) / (
                      record["owsm_cer_replace"] +
                      record["owsm_cer_equal"] +
                      record["owsm_cer_delete"]
                  )
                record["owsm_wer"] = (
                      record["owsm_wer_delete"] +
                      record["owsm_wer_replace"] +
                      record["owsm_wer_insert"]
                  ) / (
                      record["owsm_wer_replace"] +
                      record["owsm_wer_equal"] +
                      record["owsm_wer_delete"]
                  )
            data.append(record)
        except Exception as e:
            print(f"Error parsing line:\n{line}\n{e}")

# Print the beautified JSON directly.
print(json.dumps(data, indent=4))

## 2. Different Audio Domains for Codec Modeling

In this section, we will evaluate neural codec performance across three distinct audio domains, comparing how models pre-trained on specific domains perform when faced with diverse audio content:

- Speech Domain:
Speech represents highly structured audio with specific phonetic and prosodic patterns optimized for human communication. We'll test how codecs pre-trained on speech corpora (from [AMUSE speech data](https://arxiv.org/abs/2409.15897)) handle various speaking styles, accents, and recording conditions. This evaluation reveals how specialized speech encoding can achieve higher compression efficiency but may struggle with non-speech elements.

- Music Domain:
Music audio introduces unique challenges through complex harmonics, wide dynamic range, and diverse instrumental timbres. We'll examine how codecs pre-trained on music datasets (from [AMUSE music data](https://arxiv.org/abs/2409.15897))  preserve tonal quality, transients, and spatial characteristics. This comparison demonstrates the codec's ability to maintain perceptual quality of rich musical content even at low bitrates.

- General Audio Domain:
Environmental and ambient sounds encompass the broad spectrum of everyday audio beyond speech and music. We'll assess how codecs handle these diverse acoustic events, from natural sounds to mechanical noises. This evaluation tests the models' generalization capabilities and robustness when encountering audio with characteristics significantly different from their training distribution.

Additionally, we'll evaluate models trained jointly across all three domains to understand the trade-offs between specialized and generalized audio encoding approaches, providing insights into the optimal deployment strategy for various real-world applications.


We first evaluate a piece of singing voice from KiSing. Singing voice is an intersaction of speech and music. (Reference: (ACE-KiSing)[http://shijt.site/index.php/2021/05/16/kising-the-first-open-source-mandarin-singing-voice-synthesis-corpus/])

Let's take a look into how different codec models trained on different domains would have different results on this singing segments:


In [ ]:
import torch
from espnet2.bin.gan_codec_inference import AudioCoding

all_16k = AudioCoding.from_pretrained("espnet/dac_16k_all_survey")
speech_16k = AudioCoding.from_pretrained("espnet/dac_16k_speech_survey")
audio_16k = AudioCoding.from_pretrained("espnet/dac_16k_audio_survey")
music_16k = AudioCoding.from_pretrained("espnet/dac_16k_music_survey")

In [ ]:
speech, sr = sf.read("versa_demo_egs/examples/sing/ground_truth.wav")
speech = speech.astype(np.float32)
all_codec = all_16k(speech, encode_only=True)["codes"]
speech_codec = speech_16k(speech, encode_only=True)["codes"]
audio_codec = audio_16k(speech, encode_only=True)["codes"]
music_codec = music_16k(speech, encode_only=True)["codes"]

all_speech = all_16k.decode(all_codec)["resyn_audio"].squeeze(0).cpu().numpy()
speech_speech = speech_16k.decode(speech_codec)["resyn_audio"].squeeze(0).cpu().numpy()
audio_speech = audio_16k.decode(audio_codec)["resyn_audio"].squeeze(0).cpu().numpy()
music_speech = music_16k.decode(music_codec)["resyn_audio"].squeeze(0).cpu().numpy()

print("--" * 10 + " Original Speech " + "--" * 10)
display(Audio(speech, rate=sr))
librosa.display.waveshow(speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Speech Pre-trained Codec " + "--" * 10)
display(Audio(speech_speech, rate=sr))
librosa.display.waveshow(speech_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Audio Pre-trained Coddec " + "--" * 10)
display(Audio(audio_speech, rate=sr))
librosa.display.waveshow(audio_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Music Pre-trained Coddec " + "--" * 10)
display(Audio(music_speech, rate=sr))
librosa.display.waveshow(music_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Three-Domain Pre-trained Coddec " + "--" * 10)
display(Audio(all_speech, rate=sr))
librosa.display.waveshow(all_speech, sr=sr, color="blue")
plt.show()


In [ ]:
speech, sr = sf.read("versa_demo_egs/examples/sing/ground_truth.wav") # comment this out if you want to use your own voice
speech = speech.astype(np.float32)
all_codec = all_16k(speech, encode_only=True)["codes"]
speech_codec = speech_16k(speech, encode_only=True)["codes"]
audio_codec = audio_16k(speech, encode_only=True)["codes"]
music_codec = music_16k(speech, encode_only=True)["codes"]

all_speech = all_16k.decode(all_codec)["resyn_audio"].squeeze(0).cpu().numpy()
speech_speech = speech_16k.decode(speech_codec)["resyn_audio"].squeeze(0).cpu().numpy()
audio_speech = audio_16k.decode(audio_codec)["resyn_audio"].squeeze(0).cpu().numpy()
music_speech = music_16k.decode(music_codec)["resyn_audio"].squeeze(0).cpu().numpy()

print("--" * 10 + " Original Speech " + "--" * 10)
display(Audio(speech, rate=sr))
librosa.display.waveshow(speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Speech Pre-trained Codec " + "--" * 10)
display(Audio(speech_speech, rate=sr))
librosa.display.waveshow(speech_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Audio Pre-trained Coddec " + "--" * 10)
display(Audio(audio_speech, rate=sr))
librosa.display.waveshow(audio_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Music Pre-trained Coddec " + "--" * 10)
display(Audio(music_speech, rate=sr))
librosa.display.waveshow(music_speech, sr=sr, color="blue")
plt.show()

print("--" * 10 + " Reconstructed Speech from Three-Domain Pre-trained Coddec " + "--" * 10)
display(Audio(all_speech, rate=sr))
librosa.display.waveshow(all_speech, sr=sr, color="blue")
plt.show()